# Bar Charts

A **bar chart** represents categorical data with rectangular bars whose
length is proportional to the value they encode. It is one of the most
widely used chart types precisely because it is easy to read — but also
one of the most abused.

**When to use:** comparing discrete categories, ranking, showing change
between a small number of time points
**When to avoid:** too many categories (> ~15), continuous data, part-of-whole
relationships (use a pie or stacked bar instead)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="white")

co2 = pd.read_csv('CO2_selected.csv')

x_labels = co2['Country Name'].replace(['European Union', 'United States'], ['EU', 'USA'])
pos = np.arange(len(x_labels))
values = co2['2010']

## Basic bar chart

`plt.bar(positions, values)` is the minimal call. Positions are integers
here because matplotlib needs numeric x-values to control bar placement;
the country labels are added separately via `xticks`.
At this stage the chart is technically correct but hard to read:
no title, no rotated labels, bars are unlabelled.

In [ ]:
plt.figure()
plt.bar(pos, values, align='center')
plt.tight_layout()
plt.show()

## Beautified bar chart — step by step

The next cells apply a series of incremental improvements. Each one removes
ink that carries no information or adds ink that makes the data easier to read.

### Step 1 — Title, axis label, rotated ticks

Country names overlap at the default orientation.
`rotation=45` and `subplots_adjust(bottom=0.2)` give them room.

In [ ]:
plt.figure()
plt.bar(pos, values, align='center')
plt.title('CO2 emissions (metric tons per capita)')
plt.ylabel('Emission')
plt.xticks(pos, x_labels, rotation=45)
plt.subplots_adjust(bottom=0.2)
plt.tight_layout()
plt.show()

### Step 2 — Remove visual clutter: ticks and borders

The y-axis numbers are too vague to be useful at this scale — we will replace
them with direct bar labels in a later step. Removing the tick marks and all
four spines reduces chart junk without losing any information.
`tick_params` controls which tick marks and labels are visible;
iterating over `ax.spines.values()` and calling `set_visible(False)` removes
the box around the plot.

In [ ]:
plt.figure()
plt.bar(pos, values, align='center')
plt.title('CO2 emissions (metric tons per capita)')
plt.xticks(pos, x_labels, rotation=45)
plt.subplots_adjust(bottom=0.2)

plt.tick_params(top=False, bottom=False, left=False, right=False,
                labelleft=False, labelbottom=True)
for spine in plt.gca().spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

### Step 3 — Colour coding: grey baseline, highlight one bar

Painting all bars the same neutral grey (`lightslategrey`) removes the
implicit ranking that colour variation creates. A single accent colour
(`#1F77B4`) draws the eye to China — the bar we want the reader to notice.
`plt.bar()` returns a list of `Rectangle` patches, so individual bars can
be re-coloured by index after the fact.

In [ ]:
plt.figure()
bars = plt.bar(pos, values, align='center', color='lightslategrey')
bars[3].set_color('#1F77B4')   # China is at index 3

plt.title('CO2 emissions (metric tons per capita)')
plt.xticks(pos, x_labels, rotation=45)
plt.subplots_adjust(bottom=0.2)

plt.tick_params(top=False, bottom=False, left=False, right=False,
                labelleft=False, labelbottom=True)
for spine in plt.gca().spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

### Step 4 — Soften the text

Full-black labels compete with the data for attention. Setting `alpha=0.8`
on both the title and the x-tick labels shifts them to a softer grey,
keeping the bars as the visual focal point.

In [ ]:
plt.figure()
bars = plt.bar(pos, values, align='center', color='lightslategrey')
bars[3].set_color('#1F77B4')

plt.title('CO2 emissions (metric tons per capita)', alpha=0.8)
plt.xticks(pos, x_labels, rotation=45, alpha=0.8)
plt.subplots_adjust(bottom=0.2)

plt.tick_params(top=False, bottom=False, left=False, right=False,
                labelleft=False, labelbottom=True)
for spine in plt.gca().spines.values():
    spine.set_visible(False)

plt.tight_layout()
plt.show()

### Step 5 — Direct value labels on bars (final version)

Instead of a y-axis the reader has to cross-reference, we print the exact
value inside each bar. Positioning the label at `bar_height - height_shift`
keeps it inside the bar rather than floating above it, which looks cleaner.
White text (`color='w'`) ensures legibility against both the grey and blue bars.

In [ ]:
plt.figure()
bars = plt.bar(pos, values, align='center', color='lightslategrey')
bars[3].set_color('#1F77B4')

plt.title('CO2 emissions (metric tons per capita)', alpha=0.8)
plt.xticks(pos, x_labels, rotation=45, alpha=0.8)
plt.subplots_adjust(bottom=0.2)

plt.tick_params(top=False, bottom=False, left=False, right=False,
                labelleft=False, labelbottom=True)
for spine in plt.gca().spines.values():
    spine.set_visible(False)

height_shift = max(values) * 0.05
for bar in bars:
    plt.gca().text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() - height_shift,
        str(round(bar.get_height(), 1)),
        ha='center', color='w', fontsize=10
    )

plt.tight_layout()
plt.show()

# Do not cheat!

Bar charts are particularly easy to manipulate because readers implicitly
trust that the bar length encodes the full magnitude of a value.
Two common tricks can completely distort the message:

1. **Selective data** — cherry-pick the comparison group so one bar looks
   extreme relative to its neighbours
2. **Truncated y-axis** — start the y-axis above zero so that small
   absolute differences appear enormous

Both charts below use *real, unmodified numbers* — the deception is purely
in the presentation.

## Trick 1 — Selective data

By keeping only countries with emissions below 7 t/capita (dropping the USA
and Canada), China suddenly looks like the dominant emitter in the group.
The data is not falsified — the selection is.

In [ ]:
low_emitters = co2[co2['2010'] < 7].reset_index(drop=True)

plt.figure()
plt.bar(low_emitters.index, low_emitters['2010'], align='center', color='lightslategrey')
plt.title('CO2 emissions (metric tons per capita) — 2010')
plt.xticks(low_emitters.index, low_emitters['Country Name'], rotation=45)
plt.subplots_adjust(bottom=0.2)
plt.tight_layout()
plt.show()

## Trick 2 — Truncated y-axis

China (6.56 t) and the EU (7.36 t) have a difference of less than 1 tonne —
roughly 12 %. On a full-scale chart this looks like what it is: a small gap.

By setting `ylim([6.5, 7.5])` the axis no longer starts at zero. The same
0.8-tonne gap now fills most of the chart height, making the EU look *nearly
double* China's emissions. The numbers have not changed; only the visual
reference frame has.

In [ ]:
china_eu = co2[co2['Country Code'].isin(['CHN', 'EUU'])].reset_index(drop=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Honest: y-axis from zero
ax1.bar(china_eu.index, china_eu['2010'], align='center', color='lightslategrey')
ax1.set_xticks(china_eu.index)
ax1.set_xticklabels(china_eu['Country Name'], rotation=45)
ax1.set_title('Honest — axis starts at 0')
ax1.set_ylim(0, china_eu['2010'].max() * 1.15)

# Manipulated: truncated y-axis
ax2.bar(china_eu.index, china_eu['2010'], align='center', color='lightslategrey')
ax2.set_xticks(china_eu.index)
ax2.set_xticklabels(china_eu['Country Name'], rotation=45)
ax2.set_title('Misleading — axis truncated to [6.5, 7.5]')
ax2.set_ylim(6.5, 7.5)

plt.suptitle('China vs EU — CO2 emissions 2010 (same data, different y-axis)', y=1.02)
plt.tight_layout()
plt.show()